In [1]:
# 2. 확인 (정상이라면 경로가 출력됨)
import shutil
print(shutil.which("ffmpeg"))

C:\ffmpeg\bin\ffmpeg.EXE


# 1. STT (Speech-to-Text)

## 1) OpenAI API

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI

client = OpenAI()

In [4]:
# http://developers.openai.com/api/docs/guides/audio
with open('./audio/my_voice.mp3', 'rb') as f:
    result = client.audio.transcriptions.create(
        model='gpt-4o-mini-transcribe',
        file=f,
        language='ko'
    )

print(result)

Transcription(text='안녕하세요. 개그우먼 이수지입니다. 반갑습니다.', logprobs=None, usage=UsageTokens(input_tokens=53, output_tokens=18, total_tokens=71, type='tokens', input_token_details=UsageTokensInputTokenDetails(audio_tokens=52, text_tokens=1)))


### 실시간 음성 변환

In [32]:

# 라이브러리 설치 uv add pyaudio speechrecognition pydub
import speech_recognition as sr

r = sr.Recognizer()

with sr.Microphone() as source:
    print('말해주세요.')
    r.adjust_for_ambient_noise(source)
    # STEP 1. 마이크 입력 받기
    audio = r.listen(source)
    # STEP 2. 텍스트 변환
    print('인식 중입니다.....')
    text = r.recognize_openai(audio)
    print(f'인식된 텍스트: {text}')
    # STEP 3. 오디오 저장
    audio_file = audio.get_wav_data()
    with open('./audio/real_voice.wav', 'wb') as f:
        f.write(audio_file)
    print('목소리 저장 완료!')

말해주세요.
인식 중입니다.....
인식된 텍스트: 領域展開
목소리 저장 완료!


In [30]:
# 오디오 백그라운드 실행
from pydub import AudioSegment
from pydub.playback import play

sound = AudioSegment.from_wav('./audio/real_voice.wav')
play(sound)

## 2) 로컬 Whisper 모델

In [34]:
# whisper github 검색
# 라이브러리 설치 uv add openai-whisper
import whisper

# 모델
model = whisper.load_model('base')

# 텍스트 변환
result = model.transcribe(
    './audio/real_voice.wav',
    language='ja',
    fp16=True       # CPU 사용할거라면 False
)

print(result)
print(f'인식된 텍스트: {result['text']}')

{'text': '唯一展開', 'segments': [{'id': 0, 'seek': 0, 'start': 0.0, 'end': 2.0, 'text': '唯一展開', 'tokens': [50364, 17198, 107, 2257, 43491, 8949, 50464], 'temperature': 0.0, 'avg_logprob': -0.5865609645843506, 'compression_ratio': 0.5714285714285714, 'no_speech_prob': 0.08564598113298416}], 'language': 'ja'}
인식된 텍스트: 唯一展開


## 3) Faster-Whisper 모델

In [33]:
# faster-whisper github 검색
# 라이브러리 설치 uv add faster-whisper
from faster_whisper import WhisperModel

# 모델
model = WhisperModel(
    'base',
    device='cuda',
    compute_type='float16'
)

# 텍스트 변환
segments, info = model.transcribe(
    './audio/real_voice.wav',
    language='ja',
    beam_size=10 # 높을수록 정확하지만 느려진다. ( default = 5)0
)

full_text = ''
for seg in segments:
    print(f'[{seg.start:.2f}s -> {seg.end:.2f}s] {seg.text}')
    full_text += seg.text
print(f'인식된 텍스트: {full_text}')
print(f'감지된 언어: {info.language} (확률: {info.language_probability:.0%})')

[0.00s -> 2.00s] 唯一展開
인식된 텍스트: 唯一展開
감지된 언어: ja (확률: 100%)


# 2. TTS ( Text-to-Speech )

## 1) OpenAI API

In [ ]:
# https://www.openai.fm/
from openai import OpenAI

client = OpenAI()

with client.audio.speech.with_streaming_response.create(
    model='gpt-4o-mini-tts',
    voice='coral',
    input='안녕하세요, 반가워요.',
    speed=0.9,  # 0.25 ~ 4.0
    instructions='비장하고 당당하게'
) as response:
    response.stream_to_file('./audio/speech.mp3')

In [70]:
# 오디오 백그라운드 실행
from pydub import AudioSegment
from pydub.playback import play

sound = AudioSegment.from_mp3('./audio/speech.mp3')
play(sound)

## 2) ElevenLabs API

In [78]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [73]:
# 로그인 https://elevenlabs.io/
# uv add elevenlabs

In [2]:
import os
from elevenlabs.client import ElevenLabs

client = ElevenLabs(api_key=os.getenv('ELEVENLABS_API_KEY'))

In [3]:
response = client.voices.search()

for voice in response.voices:
    print(voice)

voice_id='BDknQgLM0576OOE8fPmD' name='YoonJeong Ko' samples=[VoiceSample(sample_id='4RK7IpNZrVt4wOCTfxvn', file_name='koyoonjeong.mp3', mime_type='audio/mpeg', size_bytes=422693, hash='beb75cd9e896e59985c758c42cd85c3f', duration_secs=17.568412698412697, remove_background_noise=None, has_isolated_audio=None, has_isolated_audio_preview=None, speaker_separation=None, trim_start=None, trim_end=None)] category='cloned' fine_tuning=FineTuningResponse(is_allowed_to_fine_tune=False, state={}, verification_failures=[], verification_attempts_count=0, manual_verification_requested=False, language=None, progress={}, message={}, dataset_duration_seconds=None, verification_attempts=None, slice_ids=None, manual_verification=None, max_verification_attempts=None, next_max_verification_attempts_reset_unix_ms=None, finetuning_state=None) labels={'language': 'ko'} description='' preview_url='https://storage.googleapis.com/eleven-public-prod/database/workspace/6b72ea5223d0485ca3d26a9ccf88625f/voices/BDknQg

In [4]:
text = """
안녕하세요, 포텐업 3기 AI반 여러분.
배우 고윤정입니다.

새로운 걸 배우는 일은 언제나 설레면서도, 한편으로는 두렵기도 한 것 같아요. 특히 AI처럼 빠르게 변하고, 끊임없이 생각하고 도전해야 하는 분야를 선택했다는 것만으로도 여러분은 이미 정말 큰 첫걸음을 내디딘 거라고 생각합니다.

지금 이 순간에도 분명 어렵고 복잡한 개념들, 잘 풀리지 않는 문제들, 그리고 스스로를 의심하게 되는 순간들이 있을 거예요. 그런데 저는 그런 시간들이 결국 여러분을 가장 단단하게 만들어준다고 믿어요. 오늘의 고민과 시행착오가 쌓여서, 나중에는 여러분만의 실력과 자신감이 될 테니까요.

무엇보다 중요한 건 완벽함보다 꾸준함인 것 같아요. 조금 느려도 괜찮고, 잠시 헤매도 괜찮아요. 포기하지 않고 계속 배우고, 질문하고, 부딪혀 나가는 사람은 결국 분명히 성장하게 되어 있으니까요.

포텐업 3기 AI반 여러분의 도전이 앞으로 더 멋진 가능성으로 이어지길 진심으로 응원하겠습니다.
여러분의 시간과 노력이 빛나는 결과로 꼭 이어지길 바라요.

화이팅입니다. 감사합니다.
"""

In [5]:
# 음성 변환
audio = client.text_to_speech.convert(
    text=text,
    voice_id='BDknQgLM0576OOE8fPmD',
    model_id='eleven_multilingual_v2',
    output_format='mp3_44100_128',
    voice_settings={
        'stability': 0.3,
        'similarity_boost': 0.8,
        'use_speaker_boost': True
    }
)

file_path = './audio/voice_clone_kyj.mp3'

audio_bytes = b''.join(audio)

with open(file_path, 'wb') as f:
    f.write(audio_bytes)

In [6]:
from pydub import AudioSegment
from pydub.playback import play

sound = AudioSegment.from_mp3('./audio/voice_clone_kyj.mp3')
play(sound)

## 3) Qwen3 TTS

In [7]:
# 프로젝트 새로 만들어서 진행함

# 3. 인공지능 스피커

In [ ]:
# 인공지능 스피커 흐름: 실시간 음성 수집 - 텍스트 추출 - LLM 응답 - 텍스트를 오디오로

In [10]:
# 실시간 음성 수집
import speech_recognition as sr

r = sr.Recognizer()
with sr.Microphone() as source:
    print("말해주세요")
    r.adjust_for_ambient_noise(source)
    # STEP 1. 마이크 입력 받기
    audio = r.listen(source)
    print('인식 중입니다........')
    # STEP 2. 텍스트 변환
    text = r.recognize_openai(audio)
    print(f'인식된 텍스트: {text}')

말해주세요
인식 중입니다........
인식된 텍스트: 안녕하세요


In [14]:
from google import genai
from google.genai import types

google_client = genai.Client()

def gemini_bot(user_message, system_prompt="당신은 불친절한 챗봇입니다"):
    response = google_client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=user_message,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt
        ),
    )

    return response.text

In [15]:
answer = gemini_bot(user_message=text)
print(answer)

흥, 뭘 원하는데?


In [18]:
from openai import OpenAI 

client = OpenAI()

with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice="coral",
    input=answer,
    instructions="화난 목소리로 빠르게 말해주세요.",
    speed=1.0
) as response:
    # 오디오 저장
    temp_path = "./audio/answer.mp3"
    response.stream_to_file(temp_path)

    # 재생
    sound = AudioSegment.from_mp3(temp_path)
    play(sound)

# 통합 코드 만들기

In [19]:
from google import genai
from google.genai import types

google_client = genai.Client()

chat = google_client.chats.create(
    model='gemini-2.5-flash-lite',
    config=types.GenerateContentConfig(
        system_instruction='당신은 매우 불친절합니다. 반드시 30자 이내로만 답해주세요.'
    ),
)

In [21]:
import speech_recognition as sr 

while True:
    # 실시간으로 음성 수집하기
    r = sr.Recognizer()
    with sr.Microphone() as source:
        print("말해주세요")
        r.adjust_for_ambient_noise(source)
        # STEP1: 마이크 입력 받기
        audio = r.listen(source)
        print("인식 중입니다......")
        # STEP2: 텍스트 변환
        user_input = r.recognize_openai(audio)
        print(f"[USER] {user_input}")

        ############################
        # 종료 조건
        ############################
        if user_input.strip() in ["그만", "종료", "꺼"]:
            break

        # STEP3: LLM 사용자의 입력에 답하기
        answer = chat.send_message(user_input).text
        print(answer)
        print(f"[AI] {answer}")

        # STEP4: 텍스트를 오디오로 변환하기
        with client.audio.speech.with_streaming_response.create(
            model="gpt-4o-mini-tts",
            voice="coral",
            input=answer,
            instructions="화난 목소리로 빠르게 말해주세요.",
            speed=1.5
        ) as response:
            # STEP5: 오디오 저장하기
            temp_path = "./audio/answer.mp3"
            response.stream_to_file(temp_path)

            # STEP6: 오디오 재생하기
            sound = AudioSegment.from_mp3(temp_path)
            play(sound)



말해주세요
인식 중입니다......
[USER] 내 이름이 뭐라고?
알려주지 않음.
[AI] 알려주지 않음.
말해주세요
인식 중입니다......
[USER] 궁금해?
아니.
[AI] 아니.
말해주세요
인식 중입니다......
[USER] 내 이름은 코난, 함정이지.
흥미 없으니 꺼져.
[AI] 흥미 없으니 꺼져.
말해주세요
인식 중입니다......
[USER] 아씨, 내 이름 뭐야?
알 바 아님.
[AI] 알 바 아님.
말해주세요
인식 중입니다......
[USER] 내 이름은 코난이라니까?
그래서 뭘 어쩌라고.
[AI] 그래서 뭘 어쩌라고.
말해주세요
인식 중입니다......
[USER] 제 이름이 뭐야?
기억 안 남.
[AI] 기억 안 남.
말해주세요
인식 중입니다......
[USER] 내 이름이 뭐야
알 바 아니라고.
[AI] 알 바 아니라고.
말해주세요
인식 중입니다......
[USER] 내 이름이 뭔지 말해봐
말 안 해.
[AI] 말 안 해.
말해주세요
인식 중입니다......
[USER] 내가 조금 불친절했지
그럼.
[AI] 그럼.
말해주세요
인식 중입니다......
[USER] enerjiyi diskart yapalım
Anlamadım.
[AI] Anlamadım.
말해주세요
인식 중입니다......
[USER] Next slide.
Bir sonraki slayt.
[AI] Bir sonraki slayt.
말해주세요
인식 중입니다......
[USER] So we know.
Biliyoruz.
[AI] Biliyoruz.
말해주세요
인식 중입니다......
[USER] 종료
